# Upload Models and Data to GitHub

This notebook copies trained models, results, and the Perturbed Payload dataset from Google Drive to the GitHub repo.

**What gets uploaded:**
- `models/sota/` — 6 baseline models (trained on CIC 2021)
- `models/hardened/` — 6 hardened models (retrained with Perturbed Payload)
- `results/` — experiment CSVs
- `data/perturbed_payload/` — Perturbed Payload CSVs (9 seeds) — skipped if total > 500 MB

**Requirements:** GitHub Personal Access Token with `repo` scope.

In [ ]:
# ── 1. Configuration ──────────────────────────────────────────────────────────

GITHUB_TOKEN  = ""   # your PAT here
GITHUB_USER   = "reypapin"
GITHUB_REPO   = "Dns-Tunnel-Robustness"
GITHUB_BRANCH = "main"
GIT_EMAIL     = "rleyvalao@mendozaconicet.gob.ar"
GIT_NAME      = "Reynier Leyva La O"

# Drive paths (adjust if yours differ)
DRIVE_BASE        = "/content/drive/MyDrive/Tunnel/CSV_CIC21"
DRIVE_SOTA        = f"{DRIVE_BASE}/Models_SOTA_Hybrid"
DRIVE_HARDENED    = f"{DRIVE_BASE}/Models_Hardened"
DRIVE_PERTURBED   = f"{DRIVE_BASE}/CSV_Generated"
DRIVE_RESULTS     = f"{DRIVE_BASE}/results"   # set to None if you don't have this folder

MAX_DATA_MB = 500   # skip Perturbed Payload upload if larger than this

In [ ]:
# ── 2. Mount Drive ────────────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 3. Clone repo ─────────────────────────────────────────────────────────────

import os, shutil, subprocess

REPO_URL  = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
REPO_DIR  = f"/content/{GITHUB_REPO}"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "config", "user.email", GIT_EMAIL],  cwd=REPO_DIR, check=True)
subprocess.run(["git", "config", "user.name",  GIT_NAME],   cwd=REPO_DIR, check=True)
print("Repo cloned to", REPO_DIR)

In [ ]:
# ── 4. Helper functions ───────────────────────────────────────────────────────

def folder_size_mb(path):
    """Return total size of a folder in MB."""
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if os.path.isfile(fp):
                total += os.path.getsize(fp)
    return total / (1024 ** 2)

def copy_folder(src, dst, label):
    """Copy src → dst, creating dst if needed. Returns number of files copied."""
    if not os.path.exists(src):
        print(f"[SKIP] {label}: source not found ({src})")
        return 0
    os.makedirs(dst, exist_ok=True)
    count = 0
    for fname in os.listdir(src):
        src_file = os.path.join(src, fname)
        if os.path.isfile(src_file):
            shutil.copy2(src_file, os.path.join(dst, fname))
            count += 1
    print(f"[OK]   {label}: {count} files copied  ({folder_size_mb(dst):.1f} MB)")
    return count

In [ ]:
# ── 5. Copy models ────────────────────────────────────────────────────────────

copy_folder(DRIVE_SOTA,     os.path.join(REPO_DIR, "models", "sota"),     "Models SOTA")
copy_folder(DRIVE_HARDENED, os.path.join(REPO_DIR, "models", "hardened"), "Models Hardened")

In [ ]:
# ── 6. Copy results CSVs ──────────────────────────────────────────────────────

if DRIVE_RESULTS and os.path.exists(DRIVE_RESULTS):
    copy_folder(DRIVE_RESULTS, os.path.join(REPO_DIR, "results"), "Results")
else:
    # Look for CSVs directly in DRIVE_BASE
    result_files = [f for f in os.listdir(DRIVE_BASE) if f.endswith('.csv')]
    if result_files:
        os.makedirs(os.path.join(REPO_DIR, "results"), exist_ok=True)
        for f in result_files:
            shutil.copy2(os.path.join(DRIVE_BASE, f), os.path.join(REPO_DIR, "results", f))
        print(f"[OK]   Results: {len(result_files)} CSVs copied from base dir")
    else:
        print("[SKIP] Results: no CSVs found")

In [ ]:
# ── 7. Copy Perturbed Payload CSVs (size check first) ─────────────────────────

if os.path.exists(DRIVE_PERTURBED):
    size_mb = folder_size_mb(DRIVE_PERTURBED)
    print(f"Perturbed Payload size: {size_mb:.1f} MB")
    if size_mb <= MAX_DATA_MB:
        copy_folder(DRIVE_PERTURBED,
                    os.path.join(REPO_DIR, "data", "perturbed_payload"),
                    "Perturbed Payload")
    else:
        print(f"[SKIP] Perturbed Payload exceeds {MAX_DATA_MB} MB — not uploading to GitHub.")
        print("       Consider using Git LFS or hosting on Zenodo/HuggingFace.")
else:
    print(f"[SKIP] Perturbed Payload: source not found ({DRIVE_PERTURBED})")

In [ ]:
# ── 8. Update .gitignore to track data/ and models/ ──────────────────────────

gitignore_path = os.path.join(REPO_DIR, ".gitignore")
with open(gitignore_path, "r") as f:
    content = f.read()

lines_to_remove = ["/data", "data/", "/models", "models/", "/results", "results/"]
new_lines = [l for l in content.splitlines() if l.strip() not in lines_to_remove]
with open(gitignore_path, "w") as f:
    f.write("\n".join(new_lines) + "\n")

print("[OK]   .gitignore updated")

In [ ]:
# ── 9. Commit and push ────────────────────────────────────────────────────────

subprocess.run(["git", "add", "."], cwd=REPO_DIR, check=True)

result = subprocess.run(["git", "status", "--short"], cwd=REPO_DIR, capture_output=True, text=True)
print(result.stdout)

if result.stdout.strip():
    subprocess.run(
        ["git", "commit", "-m",
         "Add trained models, results, and Perturbed Payload dataset\n\nCo-Authored-By: Claude Sonnet 4.6 <noreply@anthropic.com>"],
        cwd=REPO_DIR, check=True
    )
    subprocess.run(["git", "push", "origin", GITHUB_BRANCH], cwd=REPO_DIR, check=True)
    print("\n[DONE] Push complete.")
else:
    print("[INFO] Nothing to commit.")